In [ ]:
from torch import float32, no_grad, randn
from torch.nn import Linear, Module

import import_ipynb
from common import assert_ge, T # type: ignore
from backward import BinaryCrossEntropy, Linear, Sigmoid # type: ignore

In [ ]:
class PerceptronLog(Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = Linear(in_features, 1)
        self.sigmoid = Sigmoid()

    def forward(self, x):
        z = self.linear(x)
        p = self.sigmoid(z)
        return p


def _test_perceptron_log():
    X = randn(100, 2, dtype=float32)
    Y = (X @ T([[2.0], [-3.0]]) + T([0.5]) > 0).float()

    model = PerceptronLog(2)

    for _ in range(10):
        P = model(X)
        loss = BinaryCrossEntropy()(P, Y)
        loss.backward()

        with no_grad():
            model.linear.W -= 0.1 * model.linear.W.grad
            model.linear.b -= 0.1 * model.linear.b.grad

        model.linear.W.grad.zero_()
        model.linear.b.grad.zero_()

    accuracy = ((model(X) > 0.5) == Y).mean(dtype=float32)
    assert_ge(accuracy.item(), 0.9)


if __name__ == "__main__":
    _test_perceptron_log()